# 3.3 CSR 数据分区

## 本节学习目标

- 理解 row 与 nnz 两种平衡标准
- 追踪 Scatterv 参数

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import shutil, subprocess
for tool in ("cmake", "mpicxx", "mpirun"):
    path = shutil.which(tool)
    if path is None:
        raise RuntimeError(f"缺少必需工具：{tool}")
    print(f"{tool}: {path}")


## 分区策略

工程按 `row_ptr` 的累计 nnz 寻找边界，而非简单平均行数。这样 long-tail 矩阵中长行不会集中到单个 rank。

## 本地 CSR

rank 0 计算 row/nnz counts 和 displacements，Scatterv 分发三组数组。收到 row_ptr 后减去首个全局偏移，使本地 row_ptr 从 0 开始。

## 预期现象与结果分析

`nnz_balance_ratio=max_local_nnz/average_nnz` 越接近 1 表示计算量越均衡，但它不包含网络、内存层次和进程调度差异。

## 课后实践

给定四行 nnz 数量，手工计算两进程的合理边界并解释 row_ptr 重定位。

参考答案见 `answer/03.03_answer.md`。